# Scenario validation

Run end-to-end scenarios: take sample event logs, compute baseline metrics, run rules, and produce example insight JSON suitable for integration tests.

In [ ]:
# Make local package importable
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..').resolve()))

import json
from datetime import datetime
from src.metrics.baseline import Feeling, compute_average_post_metric, compute_count_events, MetricResult
from src.rules.evaluators import evaluate_rules_for_user

print('Imports OK')

In [ ]:
# Load sample logs from test_data and construct metric inputs
sample_path = Path('..') / 'test_data' / 'sample_users.json'
with open(sample_path, 'r') as fh:
    users = json.load(fh)

# For demo, take first user and synthesize feelings and meal times
u = users[0]
# Synthetic feelings for demo purposes
feelings = [
    Feeling(occurred_at=datetime(2026,3,1), when='post', valence=4, energy=5, stress=2),
    Feeling(occurred_at=datetime(2026,2,28), when='post', valence=3, energy=4, stress=3),
    Feeling(occurred_at=datetime(2026,2,25), when='post', valence=4, energy=5, stress=2),
]

# Compute example metrics
short = compute_average_post_metric(feelings, window_days=7, field='energy', now=datetime(2026,3,1))
long = compute_average_post_metric(feelings, window_days=30, field='energy', now=datetime(2026,3,1))

# Example aligned extras
meal_times = [datetime(2026,2,28,22,10), datetime(2026,2,25,19,30)]
morning_energies = [4, 5, 3]

metrics = {'post_energy_short': short, 'post_energy_long': long}
extra = {'meal_times': meal_times, 'morning_energies': morning_energies}

insights = evaluate_rules_for_user(u.get('id','demo_user'), metrics=metrics, extra_inputs=extra)
print('Insights:')
for it in insights:
    print(it)


Next steps:
- Expand scenario inputs from real sample logs in `test_data/`.
- Export insights to JSON and compare to expected outputs for integration tests.